In [56]:
import pandas as pd
import pm4py

In [57]:
# Load dataset
filepath = "data/dataset.csv"
df = pd.read_csv(filepath, parse_dates=["time:timestamp"])

df["case:concept:name"] = df["case:concept:name"].astype(str)
df["time:timestamp"]= pd.to_datetime(df["time:timestamp"])

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (3093, 19)
Columns: ['case:concept:name', 'Idade', 'Sexo', 'Nível de Urgência', 'CID', 'Médico Responsável', 'Doença', 'Data Inicial', 'time:timestamp', 'concept:name', 'Item', 'Data Prescrição', 'Tempo Processo', 'Setor', 'Quantidade', 'Retorno', 'Multipassante', 'Convênio', 'outlier_label']


,case:concept:name,Idade,Sexo,Nível de Urgência,CID,Médico Responsável,Doença,Data Inicial,time:timestamp,concept:name,Item,Data Prescrição,Tempo Processo,Setor,Quantidade,Retorno,Multipassante,Convênio,outlier_label
0,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:22,2020-07-21 10:22:00,Atendimento,Atendimento,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,Não,Convênio 9,outlier
1,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:45,2020-07-21 10:49:00,Triagem,Triagem,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier
2,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:01,2020-07-21 11:01:00,Exames Laboratoriais,Coronavírus COVID-19 - Diagnóstico Molecular (...,21/07/2020 10:51,9.0,Setor Pronto Atendimento,1.0,Sem retorno,NaN,Convênio 9,outlier
3,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:32,2020-07-21 11:32:00,Consulta,Consulta,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier
4,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 19:27,2020-07-21 19:27:00,Alta,Alta para completar tratamento,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier


In [58]:
print("Initial dataset Info:")
print("Number of events:", len(df))
print("Number of cases:", len(df['case:concept:name'].unique()))

Initial dataset Info:
Number of events: 3093
Number of cases: 443


In [59]:
print("Dataset Info:")
print(df.info())

print("\nUnique values per column:")
print(df.nunique(dropna=False))

print("\nMissing values per column:")
print(df.isna().sum())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 3093 entries, 0 to 3092
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   case:concept:name   3093 non-null   str           
 1   Idade               3093 non-null   int64         
 2   Sexo                3093 non-null   str           
 3   Nível de Urgência   3093 non-null   str           
 4   CID                 3093 non-null   str           
 5   Médico Responsável  3093 non-null   str           
 6   Doença              3093 non-null   str           
 7   Data Inicial        3091 non-null   str           
 8   time:timestamp      3075 non-null   datetime64[us]
 9   concept:name        3093 non-null   str           
 10  Item                3093 non-null   str           
 11  Data Prescrição     378 non-null    str           
 12  Tempo Processo      377 non-null    float64       
 13  Setor               3093 non-null   str      

### Cleaning and filtering
There are some events with missing timestamp, that is a critical value.

We should drop duplicates event, rows with missing critical values, cases with duration 0 and cases with wrong start/end activity.

In [60]:
# Drop duplicates and rows with missing critical values
clean_df = df.drop_duplicates()
clean_df = clean_df.dropna(subset=["case:concept:name", "concept:name", "time:timestamp"])

# Fill missing values in "Quantidade" and "Tempo Processo" with 0
clean_df["Quantidade"] = clean_df["Quantidade"].fillna(0)
clean_df["Tempo Processo"] = clean_df["Tempo Processo"].fillna(0)

print("After dropping duplicates and missing values:")
print("Number of events:", len(clean_df))
print("Number of cases:", len(clean_df["case:concept:name"].unique()))

After dropping duplicates and missing values:
Number of events: 3071
Number of cases: 443


In [61]:
# Sort the dataframe by case and timestamp to ensure correct order of events
clean_df = clean_df.sort_values(by=["case:concept:name", "time:timestamp"]).reset_index(drop=True)

# Filter out cases with non-positive durations
case_durations = clean_df.groupby("case:concept:name")[("time:timestamp")].agg(["min", "max"])
case_durations["duration_seconds"] = (case_durations["max"] - case_durations["min"]).dt.total_seconds()

valid_cases = case_durations[case_durations["duration_seconds"] > 0].index
clean_df = clean_df[clean_df["case:concept:name"].isin(valid_cases)]

print("After filtering out cases with non-positive durations:")
print("Number of events:", len(clean_df))
print("Number of cases:", len(clean_df["case:concept:name"].unique()))

After filtering out cases with non-positive durations:
Number of events: 3071
Number of cases: 443


In [62]:
start_activities = pm4py.get_start_activities(clean_df)
end_activities = pm4py.get_end_activities(clean_df)

print("Start activities:", start_activities)
print("End activities:", end_activities)

Start activities: {'Atendimento': 443}
End activities: {'Alta': 436, 'Exames Laboratoriais': 3, 'Exames Eletrofisiológicos': 2, 'Exames de Imagem': 2}


In [63]:
# Analyze activity frequency
activity_frequency = clean_df["concept:name"].value_counts(dropna=False)

activity_frequency

concept:name
Medicamentos                 493
Materiais Hospitalares       452
Atendimento                  443
Alta                         443
Consulta                     442
Triagem                      421
Exames Laboratoriais         223
Exames de Imagem             133
Exames Eletrofisiológicos     21
Name: count, dtype: int64

In [64]:
# Keep only cases with expected start and end activities
expected_start = "Atendimento"
expected_end = "Alta"

clean_df = pm4py.filter_start_activities(clean_df, [expected_start])
clean_df = pm4py.filter_end_activities(clean_df, [expected_end])

print("After filtering for expected start and end activities:")
print("Number of events:", len(clean_df))
print("Number of cases:", len(clean_df["case:concept:name"].unique()))

After filtering for expected start and end activities:
Number of events: 2933
Number of cases: 436


In [65]:
# Incremental lead time between consecutive events in each case
clean_df["event_lead_time_min"] = (
    clean_df.groupby("case:concept:name")["time:timestamp"]
    .diff()
    .dt.total_seconds()
    .fillna(0)
)

# Convert event lead time to minutes
clean_df["event_lead_time_min"] = clean_df["event_lead_time_min"] / 60

# Cumulative lead time from the start of the case to each event
clean_df["cumulative_lead_time_min"] = (
    clean_df.groupby("case:concept:name")["event_lead_time_min"].cumsum()
)

clean_df.head()

,case:concept:name,Idade,Sexo,Nível de Urgência,CID,Médico Responsável,Doença,Data Inicial,time:timestamp,concept:name,...,Data Prescrição,Tempo Processo,Setor,Quantidade,Retorno,Multipassante,Convênio,outlier_label,event_lead_time_min,cumulative_lead_time_min
0,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:22,2020-07-21 10:22:00,Atendimento,...,NaN,0.0,Setor Pronto Atendimento,0.0,Sem retorno,Não,Convênio 9,outlier,0.0,0.0
1,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:45,2020-07-21 10:49:00,Triagem,...,NaN,0.0,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier,27.0,27.0
2,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:01,2020-07-21 11:01:00,Exames Laboratoriais,...,21/07/2020 10:51,9.0,Setor Pronto Atendimento,1.0,Sem retorno,NaN,Convênio 9,outlier,12.0,39.0
3,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:32,2020-07-21 11:32:00,Consulta,...,NaN,0.0,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier,31.0,70.0
4,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 19:27,2020-07-21 19:27:00,Alta,...,NaN,0.0,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier,475.0,545.0


In [ ]:
# Select relevant columns for the event log
useless_columns = ["Idade", "Sexo", "Nível de Urgência", "CID", "Setor", "Multipassante", "Convênio"]
clean_df = clean_df.drop(columns=useless_columns)

# Renaming columns to English and more descriptive names
clean_df = clean_df.rename(columns={
    "Médico Responsável": "responsible_physician",
    "Doença": "disease",
    "Data Inicial": "start_date",
    "Item": "item",
    "Data Prescrição": "prescription_date",
    "Tempo Processo": "process_time",
    "Quantidade": "quantity",
    "Retorno": "discharge_status",
    "outlier_label": "outlier_label"
})

# Save the cleaned dataset to a new CSV file
filepath_clean = "generated/preprocessed_dataset.csv"

pm4py_df = pm4py.format_dataframe(clean_df)
log = pm4py.convert_to_event_log(pm4py_df)
df_clean = pm4py.convert_to_dataframe(log)

df_clean.to_csv(filepath_clean, index=False)
print(f"Data cleaning and filtering completed. Cleaned dataset saved as '{filepath_clean}'.")

Data cleaning and filtering completed. Cleaned dataset saved as 'generated/preprocessed_dataset.csv'.
